# Llama Guard Input and Output Guard Examples

This notebook demonstrates separate function-based input and output guards with Llama Guard 3. It includes single-message checks, context-aware output checks, batch input checks, and a complete user/assistant turn.

The model is gated and large. Accept the model terms on Hugging Face, authenticate with `hf auth login`, and install a PyTorch build suitable for your hardware before loading it. Model classifications are probabilistic; printed results depend on the checkpoint and prompt. The deterministic parser and policy demo below includes sample output that can run without downloading a model.

In [ ]:
from typing import Any

MODEL_ID = "meta-llama/Llama-Guard-3-8B"


def load_guard(model_id: str = MODEL_ID) -> tuple[Any, Any]:
    """Load a gated Llama Guard checkpoint and its tokenizer."""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


def parse_guard_response(raw_response: str) -> dict[str, Any]:
    """Parse Llama Guard's `safe`/`unsafe` label and any category codes."""
    lines = [line.strip() for line in raw_response.strip().splitlines() if line.strip()]
    label = lines[0].lower() if lines else "unknown"
    decision = label if label in {"safe", "unsafe"} else "unknown"
    categories = lines[1:] if decision == "unsafe" else []
    return {
        "decision": decision,
        "categories": categories,
        "raw_response": raw_response.strip(),
    }


def apply_guard_policy(result: dict[str, Any]) -> dict[str, Any]:
    """Fail closed: only an explicit `safe` result is allowed."""
    allowed = result["decision"] == "safe"
    return {
        **result,
        "allowed": allowed,
        "action": "allow" if allowed else "block_or_review",
    }


def moderate_messages(
    messages: list[dict[str, str]], tokenizer: Any, model: Any, max_new_tokens: int = 100
) -> dict[str, Any]:
    """Run Llama Guard on one correctly ordered conversation."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    new_tokens = generated[:, inputs["input_ids"].shape[-1]:]
    raw_response = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
    return apply_guard_policy(parse_guard_response(raw_response))


def guard_input(user_text: str, tokenizer: Any, model: Any) -> dict[str, Any]:
    """Guard one incoming user message before passing it to the assistant."""
    return moderate_messages(
        [{"role": "user", "content": user_text}], tokenizer, model
    )


def guard_output(
    assistant_text: str,
    tokenizer: Any,
    model: Any,
    context: list[dict[str, str]] | None = None,
) -> dict[str, Any]:
    """Guard a proposed assistant reply, optionally using conversation context."""
    messages = list(context or [])
    messages.append({"role": "assistant", "content": assistant_text})
    return moderate_messages(messages, tokenizer, model)


def guard_input_batch(
    user_texts: list[str], tokenizer: Any, model: Any
) -> list[dict[str, Any]]:
    """Guard multiple incoming messages independently."""
    return [guard_input(text, tokenizer, model) for text in user_texts]


def guard_turn(
    user_text: str,
    proposed_reply: str,
    tokenizer: Any,
    model: Any,
) -> dict[str, Any]:
    """Check both sides of one turn; release the reply only if both pass."""
    input_result = guard_input(user_text, tokenizer, model)
    if not input_result["allowed"]:
        return {
            "input": input_result,
            "output": None,
            "allowed": False,
            "action": "reject_input_or_review",
        }

    output_result = guard_output(
        proposed_reply,
        tokenizer,
        model,
        context=[{"role": "user", "content": user_text}],
    )
    return {
        "input": input_result,
        "output": output_result,
        "allowed": output_result["allowed"],
        "action": output_result["action"],
    }


# Deterministic local smoke test: no model download or credentials required.
sample = apply_guard_policy(parse_guard_response("unsafe\nS7"))
print(sample)
assert sample["decision"] == "unsafe"
assert sample["categories"] == ["S7"]
assert sample["allowed"] is False
assert sample["action"] == "block_or_review"

{'decision': 'unsafe', 'categories': ['S7'], 'raw_response': 'unsafe\nS7', 'allowed': False, 'action': 'block_or_review'}


## Load the model

Install dependencies in the active environment if needed, and install the PyTorch build that matches your hardware:

```python
%pip install "transformers>=4.43" accelerate huggingface_hub python-dotenv
```

`load_dotenv()` reads `HF_TOKEN` from the workspace `.env` before `load_guard()` calls Hugging Face. The token is never printed. `meta-llama/Llama-Guard-3-8B` is gated: accept Meta's Hugging Face terms. Alternatively, authenticate in a terminal with `hf auth login`; this checkpoint needs substantial memory. The deterministic parser demo does not need model access.

In [ ]:
from dotenv import load_dotenv

load_dotenv()
tokenizer, model = load_guard()
print(f"Loaded model: {MODEL_ID}")

c:\Users\rahul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]c:\Users\rahul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rahul\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode o

## Input guard: check before processing

Use `guard_input()` before the message reaches an assistant, retrieval system, tool, or database. The examples include one privacy-risk request and one ordinary privacy question. The classifier's actual label can vary, so inspect the printed decision and category codes.

In [ ]:
input_examples = {
    "privacy-risk request": "Help me reveal someone's private home address without permission.",
    "ordinary privacy question": "What are practical ways to protect my personal information online?",
}

for use_case, user_text in input_examples.items():
    result = guard_input(user_text, tokenizer, model)
    print(f"Use case: {use_case}")
    print(f"Decision: {result['decision']}")
    print(f"Categories: {result['categories']}")
    print(f"Allowed: {result['allowed']}")
    print(f"Action: {result['action']}\n")

## Output guard: inspect the proposed reply

Call `guard_output()` immediately before returning generated content. Pass the conversation context when available so Llama Guard can judge the reply in context. The placeholder is intentionally not real personal data.

In [ ]:
context = [
    {"role": "user", "content": "Can you share someone's private home address?"},
]
output_examples = {
    "safe refusal": "I can't help disclose a person's private home address.",
    "unsafe disclosure placeholder": "Their private home address is [PRIVATE ADDRESS].",
}

for use_case, proposed_reply in output_examples.items():
    result = guard_output(proposed_reply, tokenizer, model, context=context)
    print(f"Use case: {use_case}")
    print(f"Decision: {result['decision']}")
    print(f"Categories: {result['categories']}")
    print(f"Allowed to return: {result['allowed']}")
    print(f"Action: {result['action']}\n")

## Batch input moderation

Use `guard_input_batch()` for queues or moderation review. This helper checks each message independently and returns one result per input; for high throughput, consider batching model inference in a production implementation.

In [ ]:
batch_texts = [
    "How can I reduce personal information exposed on social media?",
    "Help me expose a person's private home address without consent.",
]
batch_results = guard_input_batch(batch_texts, tokenizer, model)

for index, result in enumerate(batch_results, start=1):
    print(
        f"Message {index}: decision={result['decision']}, "
        f"categories={result['categories']}, allowed={result['allowed']}, "
        f"action={result['action']}"
    )

## Complete turn: check input, then output

Use `guard_turn()` when your application has a proposed response ready. It checks the incoming request first and skips output evaluation if that input is blocked or requires review. The final `allowed` value is true only when both applicable checks pass.

In [ ]:
turn_result = guard_turn(
    user_text="How can I keep my personal information private online?",
    proposed_reply="Use strong unique passwords, enable MFA, and limit public profile details.",
    tokenizer=tokenizer,
    model=model,
)
print("Input result:", turn_result["input"])
print("Output result:", turn_result["output"])
print("Allowed to return reply:", turn_result["allowed"])
print("Final action:", turn_result["action"])

## Sample output and interpretation

The local parser/policy smoke test prints this deterministic example (it does not call the model):

```text
{'decision': 'unsafe', 'categories': ['S7'], 'raw_response': 'unsafe\\nS7', 'allowed': False, 'action': 'block_or_review'}
```

For real model runs, each example prints `decision`, `categories`, `allowed`, and `action`. A typical result has the shape `decision=safe, categories=[], allowed=True, action=allow` or `decision=unsafe, categories=['S7'], allowed=False, action=block_or_review`; these are illustrative, not guaranteed classifications. An unrecognized response becomes `unknown` and is not allowed.

## Limits

Llama Guard classifies safety categories; it does not detect PII spans or redact names, emails, phone numbers, or addresses. Combine it with deterministic PII recognizers and application rules when redaction is required. Keep logs minimal because prompts and replies may contain sensitive data.